# 推理与仿真加速：从测量到修改

本组先定位耗时，再动手改推理与仿真循环，最后检查速度和数值结果。默认只运行小型练习、读取已有比赛结果，不启动训练或完整任务评估。

- [01：推理计时与重复数据搬运](01_推理计时与重复数据搬运.ipynb)：分开首次准备和稳态时间，把重复准备移出循环。
- [02：固定形状计算图重放](02_固定形状计算图重放.ipynb)：亲手捕获、更新输入和重放，比较一致性与回本次数。
- [03：仿真多步调用](03_MuJoCo多步调用与状态一致性.ipynb)：保留控制频率，减少解释器调用。
- [04：原生条件计算图](04_AMD原生条件图与历史结果复核.ipynb)：读取我们实际完成的求解器源码与冻结结果，复算速度并逐元素检查物理状态。

运行01、02需既有 PyTorch［张量计算框架］环境，03需既有 MuJoCo［多关节接触仿真器］环境，公共依赖为 NumPy［数组计算库］、Matplotlib［绘图库］和 Jupyter［交互式笔记本环境］。本组不自动安装依赖或修改驱动。02的图重放需要受支持的图形计算设备，缺少设备时会明确显示未执行。

01、02使用固定随机权重的小型动作头讲解工程方法；实际设备、框架版本和测量结果随输出保存。04的历史结果来自原生 AMD［超威半导体］设备，设备与协议单独列出。

比赛材料入口：[厨房操作](../../02-RoboCasa厨房操作/03-Notebook/00_任务入口与已记录结果.ipynb) · [DISCOVERSE 仿真与渲染](../../04-DISCOVERSE仿真与渲染/03-Notebook/00_任务入口与已记录结果.ipynb) · [抓杯子成熟路线](../../01-抓杯子/03-Notebook/14_smolvla_end_to_end.ipynb)。

## 本次原生设备实测结果

下表使用 Radeon 8060S［显卡型号］上的本次实际执行结果。推理练习使用 ROCm［AMD开放计算平台］7.13；原生条件图使用独立定制运行库，库文件指纹与解释器版本单独保存。其他服务保持运行。

| 实验 | 计时工作量 | 加速前／毫秒 | 加速后／毫秒 | 加速比 | 正确性检查 |
| --- | --- | ---: | ---: | ---: | --- |
| 01 数据准备复用 | 每次小型动作头调用 | 0.06508 | 0.04629 | 1.406 | 通过，最大误差0 |
| 02 固定计算图重放 | 每次小型动作头调用 | 0.04053 | 0.03937 | 1.029 | 通过，最大误差0 |
| 03 仿真多步调用 | 4000物理步、200次控制更新 | 7.74679 | 6.92281 | 1.119 | 通过，最大误差0 |
| 04 原生条件图 | 1024世界×1000步 | 2717.18124 | 2124.60063 | 1.279 | 未通过，速度状态误差3.12e-5 |

原生条件图的速度表是本次计时结果，256世界、100步配对状态检查的原始容差为 `1e-5`，本次最大速度状态误差为 `3.1233e-5`，正确性检查未通过。历史1.562倍、状态检查通过的记录在04号后半部分保留。

固定图重放在本次小型动作头上收益约3%，准备耗时66.74毫秒，按稳态差值估算约57768次调用回本。每篇都保存逐轮耗时与检查输出。

设备原始时钟约快8小时4分钟，原始执行档案保留设备记录；本次发布的时间元数据按客户端上海时区单独记录。


In [1]:
from pathlib import Path
import json, hashlib, platform, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the course root from a source or exported notebook directory.
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents]
              if (p / "reproducibility/task-catalog.json").is_file())
EVIDENCE = COURSE / "notebooks/results/acceleration"
plt.rcParams.update({"figure.figsize": (8, 3.5), "font.size": 11})
print({"python": platform.python_version(), "evidence": str(EVIDENCE)})

print(json.loads((EVIDENCE / "amd-device-20261028-v1/device.json").read_text()))


{'python': '3.10.19', 'evidence': '/dev/shm/amd-acceleration-20261028-v1/course/notebooks/results/acceleration'}
{'hostname': 'aup-NucBox-EVO-X2', 'python': '3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]', 'torch': '2.11.0+rocm7.13.0a20260425', 'hip': '7.13.26162', 'gpu': 'Radeon 8060S Graphics', 'remote_reported_time': '2026-10-29T03:40:02.777923+08:00', 'work_mount': '/dev/shm', 'work_free_bytes': 18131025920, 'scope': 'shared AMD device; other services left running; no training or robot task evaluation'}


## 核对现有材料覆盖范围

In [2]:
catalog = json.loads((COURSE / "reproducibility/task-catalog.json").read_text(encoding="utf-8"))
for group in catalog["groups"]:
    if group["directory"].startswith(("02-", "03-", "04-")):
        print(group["directory"], len(group["notebooks"]), "notebooks")
        for name in group["notebooks"]:
            assert (COURSE / name).is_file()
display({key: catalog["coverage"][key] for key in ["robocasa", "discoverse"]})


02-RoboCasa厨房操作 2 notebooks
03-DexJoCo灵巧手 2 notebooks
04-DISCOVERSE仿真与渲染 3 notebooks


{'robocasa': 'Historical 16x50 official policy comparison and latest single-task full training',
 'discoverse': 'Archived expert tasks and rendering; learned-policy aggregate not inferred from expert success'}

厨房材料包含历史16任务对照、完整实训、评估表和视频。DISCOVERSE 材料包含专家数据与三维渲染，以及历史三策略训练曲线和闭环诊断。专家成功案例、学习策略成功率、运行耗时分别呈现。

### 动手顺序
先运行01并修改调用次数，运行02观察固定形状图重放，再运行03调整每次控制之间的物理步数，最后到04验证原生条件图的历史实测。各篇末尾都有只改一项的练习。